In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import classification_report
from sklearn.utils.class_weight import compute_class_weight
import numpy as np
import glob
import os
from google.colab import drive

# ===== 1. HARD REMOUNT =====
print("🔄 Resetting Drive Connection...")
try:
    drive.flush_and_unmount()
except:
    pass
drive.mount('/content/drive', force_remount=True)

# ===== 2. CONFIGURATION =====
BASE_ROOT = "/content/drive/MyDrive/capture24"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BATCH_SIZE = 256
NUM_WORKERS = 0  # CRITICAL: Must be 0 for Drive
MAX_EPOCHS = 20
PATIENCE = 5

# Classes
CLASSES = ['sleep', 'sit-stand', 'walking', 'mixed', 'bicycling', 'vehicle', 'unknown']
CLASS_TO_ID = {c: i for i, c in enumerate(CLASSES)}
NUM_CLASSES = len(CLASSES)
UNKNOWN_ID = CLASS_TO_ID['unknown']

# Data params
WINDOW_LEN = 1000  # 10 seconds at 100Hz
PATCH_SIZE = 20    # 0.2 second patches
NUM_PATCHES = 50   # 1000 / 20
PATCH_DIM = 60     # 20 timesteps * 3 axes

# ===== 3. LOCATE & VALIDATE FILES =====
print("🔍 Searching for data...")
found_files = glob.glob(os.path.join(BASE_ROOT, "processed_research", "*_research.npz"))
if not found_files:
    found_files = glob.glob(os.path.join(BASE_ROOT, "**", "*_research.npz"), recursive=True)

if not found_files:
    raise ValueError("❌ No files found!")

# Filter by size (avoid corrupted files)
valid_files = [f for f in found_files if os.path.getsize(f) > 1000]
print(f"✅ Found {len(valid_files)} valid files.")

# ===== 4. ROBUST DATASET =====
class MultiTaskDataset(Dataset):
    def __init__(self, file_paths, mode='train'):
        self.mode = mode
        self.data_x = []
        self.data_y = []

        print(f"   Loading {len(file_paths)} files for {mode}...")
        failed_count = 0

        for f in file_paths:
            try:
                # Load with error handling
                d = np.load(f)
                x = d['x']  # Expected: (N, 1000, 3)
                y = d['y']  # Expected: (N,)

                # Validate shape
                if x.shape[1] == WINDOW_LEN and x.shape[2] == 3:
                    # Correct reshaping process:
                    # (N, 1000, 3) → (N, 50, 20, 3) → (N, 50, 60)
                    N = x.shape[0]
                    x_temp = x.reshape(N, NUM_PATCHES, PATCH_SIZE, 3)
                    x_patches = x_temp.reshape(N, NUM_PATCHES, PATCH_DIM)

                    self.data_x.append(x_patches)
                    self.data_y.append(y)
                else:
                    failed_count += 1
                    print(f"⚠️ Skipped {os.path.basename(f)}: Wrong shape {x.shape}")

            except Exception as e:
                failed_count += 1
                print(f"⚠️ Failed {os.path.basename(f)}: {e}")

        if failed_count > 0:
            print(f"   ⚠️ {failed_count} files failed to load")

        if self.data_x:
            self.data_x = np.concatenate(self.data_x, axis=0)
            self.data_y = np.concatenate(self.data_y, axis=0)

            self.data_x = torch.tensor(self.data_x, dtype=torch.float32)
            self.data_y = torch.tensor(self.data_y, dtype=torch.long)

            print(f"   ✅ Loaded {len(self)} windows")
        else:
            raise ValueError(f"❌ No valid data loaded for {mode}!")

    def __len__(self):
        return len(self.data_x)

    def __getitem__(self, idx):
        x = self.data_x[idx]
        y = self.data_y[idx]

        # SSL mask: 15% of patches
        ssl_mask = torch.rand(NUM_PATCHES) < 0.15

        # Forecast mask: last 10 patches (2 seconds)
        fc_mask = torch.zeros(NUM_PATCHES, dtype=torch.bool)
        fc_mask[-10:] = True

        return x, ssl_mask, fc_mask, y

    def get_class_weights(self):
        labels = self.data_y.numpy()
        classes = np.unique(labels)
        weights = compute_class_weight('balanced', classes=classes, y=labels)

        w_tensor = torch.ones(NUM_CLASSES)
        for cls, w in zip(classes, weights):
            if cls < NUM_CLASSES:
                w_tensor[int(cls)] = w

        return w_tensor.to(DEVICE)

# ===== 5. MODEL =====
class ResearchTransformer(nn.Module):
    def __init__(self):
        super().__init__()
        self.emb = nn.Linear(PATCH_DIM, 256)
        self.pos = nn.Parameter(torch.randn(1, NUM_PATCHES, 256) * 0.02)
        self.encoder = nn.TransformerEncoder(
            nn.TransformerEncoderLayer(256, 8, 1024, 0.1, batch_first=True), 6)
        self.head_ssl = nn.Linear(256, PATCH_DIM)
        self.head_har = nn.Linear(256, NUM_CLASSES)

    def forward(self, x):
        h = self.emb(x) + self.pos
        h = self.encoder(h)
        return self.head_ssl(h), self.head_har(h.mean(dim=1))

# ===== 6. MAIN EXECUTION =====
if __name__ == "__main__":
    # Participant-level split
    pids = sorted(set([os.path.basename(f).split('_')[0] for f in valid_files]))
    split_idx = int(0.8 * len(pids))
    train_pids = set(pids[:split_idx])
    test_pids = set(pids[split_idx:])

    train_files = [f for f in valid_files if os.path.basename(f).split('_')[0] in train_pids]
    test_files = [f for f in valid_files if os.path.basename(f).split('_')[0] in test_pids]

    print(f"🚀 Split: {len(train_pids)} train / {len(test_pids)} test participants")
    print(f"   Files: {len(train_files)} train / {len(test_files)} test")

    # Load datasets
    train_ds = MultiTaskDataset(train_files, 'train')
    test_ds = MultiTaskDataset(test_files, 'test')

    train_loader = DataLoader(train_ds, BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)
    test_loader = DataLoader(test_ds, BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

    # Setup training
    model = ResearchTransformer().to(DEVICE)
    opt = optim.AdamW(model.parameters(), lr=1e-4)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(opt, 'min', patience=2, factor=0.5)

    class_weights = train_ds.get_class_weights()
    print(f"\n⚖️ Class Weights: {class_weights.cpu().numpy()}")

    crit_mse = nn.MSELoss()
    crit_ce = nn.CrossEntropyLoss(weight=class_weights)

    # Training loop with validation
    print("\n--- 🚀 Training Started ---")
    best_val_loss = float('inf')
    patience_counter = 0

    for epoch in range(MAX_EPOCHS):
        # Training phase
        model.train()
        train_losses = []

        for x, m_ssl, m_fc, y in train_loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            m_ssl = m_ssl.to(DEVICE)

            # Apply SSL mask correctly
            x_in = x.clone()
            x_in[m_ssl.unsqueeze(-1).expand_as(x)] = 0  # Fixed masking!

            recon, logits = model(x_in)

            # Multi-task loss
            ssl_loss = crit_mse(recon[m_ssl], x[m_ssl])
            har_loss = crit_ce(logits, y)
            loss = ssl_loss + 0.5 * har_loss

            opt.zero_grad()
            loss.backward()
            opt.step()
            train_losses.append(loss.item())

        avg_train_loss = np.mean(train_losses)

        # Validation phase
        model.eval()
        val_losses = []

        with torch.no_grad():
            for x, m_ssl, _, y in test_loader:
                x, y = x.to(DEVICE), y.to(DEVICE)
                m_ssl = m_ssl.to(DEVICE)

                x_in = x.clone()
                x_in[m_ssl.unsqueeze(-1).expand_as(x)] = 0

                recon, logits = model(x_in)
                ssl_loss = crit_mse(recon[m_ssl], x[m_ssl])
                har_loss = crit_ce(logits, y)
                val_losses.append((ssl_loss + 0.5 * har_loss).item())

        avg_val_loss = np.mean(val_losses)
        scheduler.step(avg_val_loss)

        print(f"Epoch {epoch+1}/{MAX_EPOCHS} | Train: {avg_train_loss:.4f} | Val: {avg_val_loss:.4f} | LR: {opt.param_groups[0]['lr']:.2e}")

        # Early stopping
        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            torch.save(model.state_dict(), "best_model.pth")
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= PATIENCE:
                print("🛑 Early stopping triggered")
                break

    # Final evaluation
    print("\n--- 📊 Final Evaluation ---")
    model.load_state_dict(torch.load("best_model.pth"))
    model.eval()

    y_true, y_pred = [], []
    fc_errors = {'0.5s': [], '1.0s': [], '2.0s': []}

    with torch.no_grad():
        for x, _, m_fc, y in test_loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            m_fc = m_fc.to(DEVICE)

            # Forecasting: mask last 10 patches
            x_in = x.clone()
            x_in[m_fc.unsqueeze(-1).expand_as(x)] = 0

            recon, logits = model(x_in)

            # HAR predictions
            y_true.extend(y.cpu().numpy())
            y_pred.extend(logits.argmax(1).cpu().numpy())

            # Forecast errors at different horizons
            diff = torch.abs(recon - x)
            fc_errors['0.5s'].append(diff[:, -10, :].mean().item())  # First masked patch
            fc_errors['1.0s'].append(diff[:, -5, :].mean().item())   # Middle
            fc_errors['2.0s'].append(diff[:, -1, :].mean().item())   # Last masked patch

    # Print results
    print("\n1. Forecasting MAE (Multi-Horizon):")
    for horizon, errors in fc_errors.items():
        print(f"   {horizon}: {np.mean(errors):.4f}")

    print("\n2. HAR Classification (Known Classes Only):")
    mask = np.array(y_true) != UNKNOWN_ID

    if mask.sum() > 0:
        y_true_clean = np.array(y_true)[mask]
        y_pred_clean = np.array(y_pred)[mask]

        print(classification_report(
            y_true_clean,
            y_pred_clean,
            labels=list(range(NUM_CLASSES-1)),
            target_names=CLASSES[:-1],
            zero_division=0
        ))

        # Overall accuracy
        acc = (y_pred_clean == y_true_clean).mean()
        print(f"\nOverall Accuracy (Known): {acc*100:.2f}%")
    else:
        print("⚠️ Warning: All predictions were 'unknown'")

    print("\n✅ Training and Evaluation Complete!")


🔄 Resetting Drive Connection...
Drive not mounted, so nothing to flush and unmount.
Mounted at /content/drive
🔍 Searching for data...
✅ Found 151 valid files.
🚀 Split: 120 train / 31 test participants
   Files: 120 train / 31 test
   Loading 120 files for train...
   ✅ Loaded 1124194 windows
   Loading 31 files for test...
   ✅ Loaded 279621 windows

⚖️ Class Weights: [ 0.58213615  0.5409327   3.4779034   1.9178307  22.011944    5.652511
  0.41627887]

--- 🚀 Training Started ---
Epoch 1/20 | Train: 0.6172 | Val: 0.6153 | LR: 1.00e-04
Epoch 2/20 | Train: 0.5729 | Val: 0.6285 | LR: 1.00e-04
Epoch 3/20 | Train: 0.5568 | Val: 0.6078 | LR: 1.00e-04
Epoch 4/20 | Train: 0.5466 | Val: 0.6235 | LR: 1.00e-04
Epoch 5/20 | Train: 0.5373 | Val: 0.6094 | LR: 1.00e-04
Epoch 6/20 | Train: 0.5295 | Val: 0.6015 | LR: 1.00e-04
Epoch 7/20 | Train: 0.5226 | Val: 0.6060 | LR: 1.00e-04
Epoch 8/20 | Train: 0.5162 | Val: 0.6050 | LR: 1.00e-04
Epoch 9/20 | Train: 0.5099 | Val: 0.6069 | LR: 5.00e-05
Epoch 10/20 

KeyboardInterrupt: 

In [ ]:
# 4. DATASET (Robust Version)
class MultiTaskDataset(Dataset):
    def __init__(self, file_paths, mode):
        self.mode = mode
        self.data_x = []
        self.data_y = []

        print(f"   Loading {len(file_paths)} files for {mode}...")
        for i, f in enumerate(file_paths):
            try:
                # Load with mmap_mode='r' to test read, then copy to RAM
                with np.load(f) as d:
                    x = d['x']
                    y = d['y']

                # Validate shape (1000 time steps -> 50 patches)
                if x.shape[1] * x.shape[2] == 50 * 60:
                    self.data_x.append(x.reshape(x.shape[0], 50, 60))
                    self.data_y.append(y)
            except Exception as e:
                print(f"⚠️ Failed to load {os.path.basename(f)}: {e}")

        if self.data_x:
            self.data_x = np.concatenate(self.data_x, axis=0)
            self.data_y = np.concatenate(self.data_y, axis=0)

            self.data_x = torch.tensor(self.data_x, dtype=torch.float32)
            self.data_y = torch.tensor(self.data_y, dtype=torch.long)
        else:
            raise ValueError(f"No valid data loaded for {mode} set!")

        print(f"   ✅ Loaded {len(self.data_x)} windows for {mode}.")

    def __len__(self): return len(self.data_x)

    def __getitem__(self, idx):
        x = self.data_x[idx]
        y = self.data_y[idx]
        ssl_mask = torch.rand(50) < 0.15
        return x, ssl_mask, y

    def get_class_weights(self):
        labels = self.data_y.numpy()
        classes = np.unique(labels)
        weights = compute_class_weight(class_weight='balanced', classes=classes, y=labels)
        w_tensor = torch.ones(NUM_CLASSES)
        for cls, w in zip(classes, weights):
            if cls < NUM_CLASSES: w_tensor[cls] = w
        return w_tensor.to(DEVICE)